# 99 - Herramientas adicionales para IA: notebook integrador

Este notebook reúne ejemplos mínimos de varias herramientas útiles en programación de IA sin crear un notebook separado para cada una.

La idea es entender el patrón de trabajo antes de depender de una librería concreta. Cuando una herramienta externa no está instalada, se usa una versión simulada con Python estándar.

## Mapa rápido

1. `LlamaIndex`, `Haystack`, `Chroma`, `FAISS` y `Qdrant`: recuperación de documentos y RAG.
2. `LangGraph`: flujos con estado, decisiones y ciclos.
3. `PydanticAI` e `Instructor`: salidas estructuradas y validación.
4. `LiteLLM` y `Ollama`: cambiar de proveedor o usar modelos locales.
5. `Ragas`: evaluación de respuestas RAG.
6. `MLflow`: registro de experimentos y métricas.

## Preparación para Google Colab

Ejecuta la siguiente celda al principio si abres este notebook en Colab. Detecta el entorno e instala solo las librerías externas que falten.

Los ejemplos principales del notebook siguen funcionando aunque no instales estas librerías, porque están implementados con Python estándar para facilitar la práctica en clase.

Esta celda detecta si el notebook se está ejecutando en Google Colab. En ese caso instala las librerías opcionales que falten; en local solo muestra el comando de instalación manual.


In [ ]:
import importlib.util
import os
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules or bool(os.environ.get("COLAB_RELEASE_TAG"))

OPTIONAL_PACKAGES = {
    "llama_index": "llama-index",
    "haystack": "haystack-ai",
    "chromadb": "chromadb",
    "faiss": "faiss-cpu",
    "qdrant_client": "qdrant-client",
    "langgraph": "langgraph",
    "pydantic_ai": "pydantic-ai",
    "instructor": "instructor",
    "litellm": "litellm",
    "ollama": "ollama",
    "ragas": "ragas",
    "mlflow": "mlflow",
}

def install_if_missing(import_name, package_name):
    if importlib.util.find_spec(import_name) is not None:
        print(f"OK: {package_name} ya está instalado")
        return
    print(f"Instalando {package_name}...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package_name])

if IN_COLAB:
    for import_name, package_name in OPTIONAL_PACKAGES.items():
        install_if_missing(import_name, package_name)
    print("Instalación de dependencias opcionales completada en Colab.")
else:
    print("No se ha detectado Colab. Se omite la instalación automática.")
    print("Si quieres instalar todo manualmente: pip install llama-index haystack-ai chromadb faiss-cpu qdrant-client langgraph pydantic-ai instructor litellm ollama ragas mlflow")

Esta celda carga utilidades de Python estándar y define el corpus documental común que usarán los ejemplos de RAG, grafos, evaluación y tracking.


In [ ]:
from dataclasses import dataclass, asdict
from collections import Counter
from math import sqrt
import json
import re

DOCUMENTS = [
    {"id": "doc1", "title": "LlamaIndex", "text": "LlamaIndex ayuda a construir aplicaciones RAG centradas en documentos propios."},
    {"id": "doc2", "title": "LangGraph", "text": "LangGraph permite definir grafos con estado para agentes y flujos con decisiones."},
    {"id": "doc3", "title": "PydanticAI", "text": "PydanticAI usa modelos Pydantic para validar salidas estructuradas de agentes."},
    {"id": "doc4", "title": "Ollama", "text": "Ollama ejecuta modelos de lenguaje en local sin depender de una API externa."},
    {"id": "doc5", "title": "Ragas", "text": "Ragas sirve para evaluar sistemas RAG con métricas de relevancia del contexto y fidelidad de la respuesta."},
    {"id": "doc6", "title": "MLflow", "text": "MLflow registra experimentos, parámetros, métricas y versiones de modelos."},
]

def tokenize(text):
    return re.findall(r"[a-záéíóúñü]+", text.lower())

def cosine_counter(a, b):
    common = set(a) & set(b)
    numerator = sum(a[t] * b[t] for t in common)
    norm_a = sqrt(sum(v * v for v in a.values()))
    norm_b = sqrt(sum(v * v for v in b.values()))
    return numerator / (norm_a * norm_b) if norm_a and norm_b else 0.0

print(f"Documentos cargados: {len(DOCUMENTS)}")

## 1. RAG con estilo LlamaIndex, Haystack y bases vectoriales

Las herramientas cambian, pero el patrón suele ser el mismo: documentos, fragmentación, embeddings o representación vectorial, recuperación y respuesta fundamentada.

Esta celda implementa un recuperador vectorial mínimo con conteo de tokens y similitud coseno. Sirve para simular el papel de una base vectorial en un sistema RAG.


In [ ]:
class MiniVectorStore:
    def __init__(self, documents):
        self.documents = documents
        self.vectors = [Counter(tokenize(doc["title"] + " " + doc["text"])) for doc in documents]

    def search(self, query, top_k=3):
        query_vector = Counter(tokenize(query))
        scored = []
        for doc, vector in zip(self.documents, self.vectors):
            scored.append((cosine_counter(query_vector, vector), doc))
        scored.sort(key=lambda item: item[0], reverse=True)
        return [{"score": round(score, 3), **doc} for score, doc in scored[:top_k] if score > 0]

def answer_with_context(question, contexts):
    if not contexts:
        return "No tengo contexto suficiente para responder."
    titles = ", ".join(ctx["title"] for ctx in contexts)
    facts = " ".join(ctx["text"] for ctx in contexts)
    return f"Pregunta: {question}\nRespuesta simulada: {facts}\nFuentes: {titles}"

store = MiniVectorStore(DOCUMENTS)
question = "¿Qué herramienta usarías para evaluar un sistema RAG?"
contexts = store.search(question, top_k=2)
print(json.dumps(contexts, indent=2, ensure_ascii=False))
print(answer_with_context(question, contexts))

Equivalencias aproximadas:

1. `MiniVectorStore` se parece a una base vectorial mínima (`Chroma`, `FAISS`, `Qdrant`).
2. `search` representa la fase de recuperación de `LlamaIndex` o `Haystack`.
3. `answer_with_context` representa la generación final de una respuesta RAG.

## 2. Flujo con estado estilo LangGraph

Un grafo permite decidir el siguiente paso según el estado. Aquí no usamos `langgraph`, pero sí su idea principal: nodos, estado y rutas condicionales.

Esta celda define un flujo con estado inspirado en LangGraph: clasifica la intención, decide la ruta y ejecuta recuperación o respuesta alternativa según el caso.


In [ ]:
def classify_intent(state):
    text = state["question"].lower()
    if "eval" in text or "rag" in text:
        state["intent"] = "rag"
    elif "local" in text or "ollama" in text:
        state["intent"] = "local_model"
    else:
        state["intent"] = "general"
    return state

def retrieve_node(state):
    state["contexts"] = store.search(state["question"], top_k=2)
    return state

def answer_node(state):
    state["answer"] = answer_with_context(state["question"], state.get("contexts", []))
    return state

def fallback_node(state):
    state["answer"] = "Respuesta general: revisa la documentación de la herramienta y construye un prototipo pequeño."
    return state

def run_graph(question):
    state = {"question": question, "trace": []}
    for node in [classify_intent]:
        state = node(state)
        state["trace"].append(node.__name__)
    if state["intent"] == "rag":
        for node in [retrieve_node, answer_node]:
            state = node(state)
            state["trace"].append(node.__name__)
    else:
        state = fallback_node(state)
        state["trace"].append(fallback_node.__name__)
    return state

result = run_graph("Quiero evaluar un RAG con métricas")
print(json.dumps(result, indent=2, ensure_ascii=False))

## 3. Salidas estructuradas estilo PydanticAI e Instructor

En aplicaciones reales no basta con texto libre. Muchas veces necesitamos un objeto validado con campos concretos.

Esta celda define un esquema estructurado para recomendaciones de herramientas y valida que una salida simulada del modelo cumpla el contrato esperado.


In [ ]:
@dataclass
class ToolRecommendation:
    tool: str
    category: str
    reason: str
    confidence: float

def validate_recommendation(data):
    required = {"tool", "category", "reason", "confidence"}
    missing = required - set(data)
    if missing:
        raise ValueError(f"Faltan campos: {sorted(missing)}")
    confidence = float(data["confidence"])
    if not 0 <= confidence <= 1:
        raise ValueError("confidence debe estar entre 0 y 1")
    return ToolRecommendation(
        tool=str(data["tool"]),
        category=str(data["category"]),
        reason=str(data["reason"]),
        confidence=confidence,
    )

raw_model_output = {
    "tool": "Ragas",
    "category": "evaluación",
    "reason": "Permite medir si la respuesta está fundamentada en el contexto recuperado.",
    "confidence": 0.86,
}
recommendation = validate_recommendation(raw_model_output)
print(asdict(recommendation))

Si se instalara `pydantic`, esta validación se expresaría con un `BaseModel`. `PydanticAI` e `Instructor` aplican esta idea a respuestas de LLMs.

## 4. Proveedores de modelos estilo LiteLLM y Ollama

Conviene aislar el código de negocio del proveedor concreto. Así se puede cambiar entre API cloud, modelo local o simulación.

Esta celda centraliza la llamada a modelos simulados para mostrar cómo aislar el código del proveedor concreto, igual que haríamos con LiteLLM u Ollama.


In [ ]:
def call_model(prompt, provider="mock", model="demo"):
    if provider == "ollama":
        return f"[Ollama/{model}] Respuesta local simulada para: {prompt}"
    if provider == "cloud":
        return f"[Cloud/{model}] Respuesta remota simulada para: {prompt}"
    return f"[Mock/{model}] Respuesta determinista para clase: {prompt}"

for provider in ["mock", "ollama", "cloud"]:
    print(call_model("Resume para qué sirve LiteLLM", provider=provider, model="llm-demo"))

`LiteLLM` proporciona una interfaz común para muchos proveedores. `Ollama` permite ejecutar modelos locales, por ejemplo con una llamada HTTP a `http://localhost:11434`.

## 5. Evaluación estilo Ragas

Para evaluar RAG se pueden medir señales como relevancia del contexto, uso de fuentes y fidelidad de la respuesta. Aquí usamos métricas simples para entender el concepto.

Esta celda calcula métricas sencillas para evaluar un ejemplo RAG: relevancia del contexto, fidelidad aproximada y presencia de citas.


In [ ]:
def evaluate_rag(question, answer, contexts):
    question_terms = set(tokenize(question))
    context_terms = set(tokenize(" ".join(ctx["text"] for ctx in contexts)))
    answer_terms = set(tokenize(answer))
    context_relevance = len(question_terms & context_terms) / max(1, len(question_terms))
    faithfulness = len(answer_terms & context_terms) / max(1, len(answer_terms))
    cites_sources = all(ctx["title"] in answer for ctx in contexts[:1])
    return {
        "context_relevance": round(context_relevance, 3),
        "faithfulness_proxy": round(faithfulness, 3),
        "cites_sources": cites_sources,
    }

answer = answer_with_context(question, contexts)
metrics = evaluate_rag(question, answer, contexts)
print(json.dumps(metrics, indent=2, ensure_ascii=False))

## 6. Tracking estilo MLflow

MLflow permite guardar parámetros, métricas y artefactos. Aquí se crea un registro mínimo en JSON para enseñar la idea sin instalar nada.

Esta celda registra parámetros, métricas y una muestra de entrada/salida en un JSON, simulando el tipo de información que guardaríamos con MLflow.


In [ ]:
experiment_run = {
    "name": "rag_tools_demo",
    "params": {
        "retriever": "MiniVectorStore",
        "top_k": 2,
        "provider": "mock",
    },
    "metrics": metrics,
    "sample": {
        "question": question,
        "contexts": [ctx["title"] for ctx in contexts],
    },
}

with open("mlflow_like_run.json", "w", encoding="utf-8") as file:
    json.dump(experiment_run, file, indent=2, ensure_ascii=False)

print(json.dumps(experiment_run, indent=2, ensure_ascii=False))

## Resumen de equivalencias

| Necesidad | Herramientas | Ejemplo del notebook |
|---|---|---|
| Consultar documentos propios | `LlamaIndex`, `Haystack` | `MiniVectorStore.search` |
| Guardar y buscar vectores | `Chroma`, `FAISS`, `Qdrant` | Contadores de tokens y coseno |
| Flujos con estado | `LangGraph` | `run_graph` con rutas condicionales |
| JSON fiable desde LLMs | `PydanticAI`, `Instructor` | `validate_recommendation` |
| Cambiar proveedor de modelo | `LiteLLM`, `Ollama` | `call_model` |
| Evaluar RAG | `Ragas` | `evaluate_rag` |
| Registrar experimentos | `MLflow` | `experiment_run` |

## Reto adicional

1. Añade tres documentos nuevos sobre herramientas de IA.
2. Cambia la métrica de recuperación para que dé más peso al título.
3. Añade una ruta nueva al grafo para preguntas sobre despliegue.
4. Guarda dos ejecuciones distintas y compara sus métricas como si fueran runs de MLflow.

## Para profundizar
Este notebook introduce Herramientas IA adicionales. La documentación completa incluye:
- RAG, LangGraph, PydanticAI, LiteLLM, Ragas

Consulta el documento correspondiente en Documentacion/.

In [ ]:
# Los ejemplos avanzados están en los documentos de Documentacion/# Consulta el archivo correspondiente según lo que quieras profundizar.

## Stack completo
Este notebook muestra herramientas principais. Para un **stack completo** añade:
- **FastAPI** (`FastAPI_Documentacion.md`): APIs profesionais que escalan, en lugar de Gradio.
- **Ollama** (`Ollama_Documentacion.md`): modelos locais sen custo de API.

Consulta os documentos completos en `Documentacion/`.

In [ ]:
# Stack completo:
# - Gradio (demos) -> FastAPI (produción)
# - OpenAI API -> Ollama (local)
# - MLflow (tracking)
# - LangChain + LlamaIndex (RAG)
# - DSPy (optimización)
# 
# Consulta os documentos adicionais en Documentacion/